In [ ]:
!pip install -q opencv-python-headless pillow tqdm transformers torch torchvision scikit-learn matplotlib seaborn

import os
import cv2
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

from transformers import CLIPProcessor, CLIPModel

from google.colab import files

print("Upload 2–3 short videos")
uploaded = files.upload()

video_paths = list(uploaded.keys())
print("Videos:", video_paths)

In [ ]:
FRAMES_DIR = "frames"
os.makedirs(FRAMES_DIR, exist_ok=True)

def extract_frames(video_path, video_id, interval_sec=2):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    interval_frames = int(fps * interval_sec)

    metadata = []
    frame_count, saved_count = 0, 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % interval_frames == 0:
            timestamp = frame_count / fps
            fname = f"{video_id}_frame_{saved_count}.jpg"
            fpath = os.path.join(FRAMES_DIR, fname)
            cv2.imwrite(fpath, frame)

            metadata.append([video_id, timestamp, fpath])
            saved_count += 1

        frame_count += 1

    cap.release()
    return metadata

all_meta = []
for i, vp in enumerate(video_paths):
    all_meta.extend(extract_frames(vp, f"video{i+1}"))

df_meta = pd.DataFrame(all_meta, columns=["video_id","timestamp","file_path"])
df_meta.to_csv("metadata.csv", index=False)
df_meta.head()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

embeddings = []

for path in tqdm(df_meta["file_path"]):
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

        # ensure tensor (handles HF versions)
        if hasattr(outputs, "pooler_output"):
            emb = outputs.pooler_output
        else:
            emb = outputs

        emb = emb / emb.norm(p=2, dim=-1, keepdim=True)

    embeddings.append(emb.cpu().numpy()[0])

embeddings = np.array(embeddings)

assert embeddings.shape[0] == len(df_meta)
assert not np.isnan(embeddings).any()

In [ ]:
# identical image test
sim_test = cosine_similarity([embeddings[0]], [embeddings[0]])[0][0]
print("Self similarity:", sim_test)
assert sim_test > 0.99

sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(10,8))
sns.heatmap(sim_matrix, cmap="viridis")
plt.title("Frame Similarity Heatmap")
plt.show()

In [ ]:
def retrieve_similar(query_idx, top_k=5):
    sims = sim_matrix[query_idx]
    idxs = np.argsort(sims)[::-1][1:top_k+1]
    return idxs, sims[idxs]

query_frames = [0, len(df_meta)//2, len(df_meta)-1]

for q in query_frames:
    idxs, scores = retrieve_similar(q)

    print(f"\nQuery Frame {q}")
    display(Image.open(df_meta.iloc[q]["file_path"]))

    for i, s in zip(idxs, scores):
        print(f"Similarity: {s:.3f}")
        display(Image.open(df_meta.iloc[i]["file_path"]))
